In [1]:
from typing import Literal

from dotenv import load_dotenv

_=load_dotenv()  # Load environment variables from .env file

In [ ]:
### Tools- Internet search
import os

from tavily import TavilyClient

tavily_client=TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def web_search(query:str,max_results:int=5,
topic: Literal["general","sports","news","finance"]="general",
include_raw_content:bool=False):
    """Run a web search"""
    return tavily_client.search(query,
    max_results=max_results,include_raw_content=include_raw_content,topic=topic)

## State  Backend

In [2]:
import os

from deepagents import create_deep_agent
from deepagents.backends import StateBackend


In [3]:
# -----------------------------------------------------------------------------
# 1. Create the agent — these two are equivalent
# -----------------------------------------------------------------------------
agent = create_deep_agent(model="openai:gpt-5.4")

# Under the hood this is what `agent` is doing — explicit StateBackend:
agent2 = create_deep_agent(
    model="openai:gpt-5.4",
    backend=StateBackend(),
)

In [4]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
#    (StateBackend keeps that file inside LangGraph state)
# -----------------------------------------------------------------------------
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

In [5]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='b3193f81-f354-4824-9785-f223d42c35a3'),
  AIMessage(content=[{'arguments': '{"file_path":"/notes/todo.txt","content":"1. Record video\\n2. Edit video\\n3. Upload video"}', 'call_id': 'call_NPUjCRVALfagSh45Gv48zFo6', 'name': 'write_file', 'type': 'function_call', 'id': 'fc_03e2c2af0655b7ae006a59068520b8819db91499379c82954c', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_03e2c2af0655b7ae006a59068491e4819d8e6cba48c0bb4fb9', 'created_at': 1784219268.0, 'metadata': {}, 'model': 'gpt-5.4-2026-03-05', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05'}, id='resp_03e2c2af0655b7ae006a59068491e4819d8e6cba48c0bb4fb9', tool_calls=[{'name': 'write_file', 'args': {

In [6]:
# The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'Created `/notes/todo.txt` with the requested content.', 'annotations': [], 'id': 'msg_03e2c2af0655b7ae006a5906872cfc819da5ce22385d6c569a', 'phase': 'final_answer'}]


In [7]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
# -----------------------------------------------------------------------------
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-07-16T16:27:50.180115+00:00', 'modified_at': '2026-07-16T16:27:50.180115+00:00'}


In [8]:
# -----------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# -----------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_03e2c2af0655b7ae006a590744f04c819d82dc69ad3fa26618', 'phase': 'final_answer'}]


## FileSystemBackend

In [9]:
# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model="openai:gpt-5.4",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [10]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
# -----------------------------------------------------------------------------
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'Created `/notes/todo.txt` with the requested content.', 'annotations': [], 'id': 'msg_09420b106dacfb6d006a5907c700dc819280f0117fc1e9b98f', 'phase': 'final_answer'}]


In [11]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path

print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print("✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 /Users/eshantdas/Desktop/SelfStudy/PersonalTest/DeepAgents_2pointo/deep_agentsDemo/notes/todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [13]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model="openai:gpt-5.4",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_0a1085126b6e583d006a59084bdb4c81a3bafe509c42d626f3', 'phase': 'final_answer'}]


## Deep Agent — StoreBackend verification

Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do. """

In [14]:

from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model="openai:gpt-5.4",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [15]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
[{'type': 'text', 'text': 'Created `/notes/todo.txt` with the requested content.', 'annotations': [], 'id': 'msg_0d53b06dc2486927006a59085fb46081a0ac47a2d1041adf57', 'phase': 'final_answer'}]


In [16]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
# We can read from a different thread as well
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_02eb60ea8339f1cf006a59088b3ce481a0928093c110d70228', 'phase': 'final_answer'}]


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

## <font color='red'>Composite Backend and Context Backend</font>

- Explore on your own